# Notebook 06 â€” On-Device LLM (Phi-3.5 Mini INT4)

Downloads Phi-3.5 Mini GGUF, tests World Model scene descriptions, benchmarks latency (target <400ms).

## Cell 06-01 | Mount & Install llama-cpp-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, time

BASE   = '/content/drive/MyDrive/ARGUS'
MODELS = f'{BASE}/models/llm'
os.makedirs(MODELS, exist_ok=True)

# Install CPU wheel (pre-built, no compilation).
# GPU inference runs on the Jetson — we only need llama-cpp-python here
# to download and smoke-test the GGUF before transferring to Jetson.
print('Installing llama-cpp-python (CPU, pre-built wheel)...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'])
print('llama-cpp-python installed')


## Cell 06-02 | Download Phi-3.5 Mini GGUF

In [ ]:
# hf_hub_download resumes partial downloads via HTTP range requests.
# We verify the file is >= 2 GB before accepting it.
from huggingface_hub import hf_hub_download
import os

LLM_DIR   = f'{MODELS}'
LLM_FNAME = 'Phi-3.5-mini-instruct-Q4_K_M.gguf'
LLM_PATH  = f'{LLM_DIR}/{LLM_FNAME}'
MIN_GB    = 2.0

def _gguf_ok(p): return os.path.exists(p) and os.path.getsize(p)/1e9 >= MIN_GB

if _gguf_ok(LLM_PATH):
    print(f'✓ {LLM_FNAME} ready ({os.path.getsize(LLM_PATH)/1e9:.2f} GB)')
else:
    if os.path.exists(LLM_PATH):
        size_gb = os.path.getsize(LLM_PATH)/1e9
        print(f'Partial GGUF detected ({size_gb:.2f}/{MIN_GB} GB) — removing for clean download')
        os.remove(LLM_PATH)
    print('Downloading Phi-3.5-mini Q4_K_M (~2.2 GB) ...')
    hf_hub_download(
        repo_id='bartowski/Phi-3.5-mini-instruct-GGUF',
        filename=LLM_FNAME,
        local_dir=LLM_DIR,
    )

if _gguf_ok(LLM_PATH):
    print(f'✅ {LLM_FNAME} ({os.path.getsize(LLM_PATH)/1e9:.2f} GB)')
else:
    print('⚠ GGUF still incomplete — re-run this cell to resume')

## Cell 06-03 | Load LLM & Define ARGUS System Prompt

In [ ]:
from llama_cpp import Llama
import time, json

LLM_PATH = f'{MODELS}/Phi-3.5-mini-instruct-Q4_K_M.gguf'
print('Loading Phi-3.5 Mini...')
llm = Llama(model_path=LLM_PATH, n_ctx=2048, n_gpu_layers=-1, n_threads=4, verbose=False)
print('Phi-3.5 Mini loaded')

ARGUS_SYSTEM = (
    'You are ARGUS, an AI assistant embedded in smart glasses for a blind person. '
    'You receive structured scene data and convert it to clear, natural spoken descriptions.\n\n'
    'Rules:\n'
    '1. Keep responses SHORT (1-2 sentences max, unless asked for detail)\n'
    '2. NEVER describe objects marked private=true\n'
    '3. Always mention safety hazards (stairs, drop) FIRST\n'
    '4. Give distances and directions clearly\n'
    '5. Speak naturally, not robotically\n\n'
    'Scene data format:\n'
    '[Object: label, Distance: Xm, Direction: left|center|right|rear, Private: true|false]\n'
    'Hazard field: null | stairs_down | stairs_up | drop | door\n\n'
    'Respond ONLY with the spoken description. No preamble.'
)
print('System prompt defined')
print(f'Prompt length: {len(ARGUS_SYSTEM)} characters')

## Cell 06-04 | World Model â†’ Scene Description (Core Function)

In [ ]:
import json, time

def world_model_to_speech(world_model: dict, user_query=None):
    # Convert World Model dict to spoken ARGUS response
    objects = world_model.get('objects', [])
    hazard  = world_model.get('hazard')
    floor   = world_model.get('navigable_floor', True)
    nearest = world_model.get('nearest_obstacle_dist', 999)

    visible   = [o for o in objects if not o.get('private', False)]
    scene_str = 'SCENE DATA:\n'
    if hazard:
        scene_str += f'  HAZARD: {hazard}\n'
    if not floor:
        scene_str += '  No clear floor detected\n'
    scene_str += f'  Nearest obstacle: {nearest:.1f}m\n'
    for obj in visible[:8]:
        scene_str += f'  [Object: {obj["label"]}, Distance: {obj["distance"]:.1f}m, Direction: {obj["direction"]}]\n'

    user_msg = f'{scene_str}\nUser asks: {user_query}' if user_query else f'{scene_str}\nProvide brief navigation guidance.'
    prompt   = f'<|system|>\n{ARGUS_SYSTEM}<|end|>\n<|user|>\n{user_msg}<|end|>\n<|assistant|>\n'

    t0  = time.time()
    out = llm(prompt, max_tokens=80, temperature=0.1, stop=['<|end|>', '<|user|>', '\n\n'], echo=False)
    return out['choices'][0]['text'].strip(), (time.time() - t0) * 1000

print('World Model query function defined')

## Cell 06-05 | Test Scenarios

In [ ]:
test_scenarios = [
    {'name': 'Simple room', 'query': 'what is in front of me?',
     'world': {'objects': [{'label': 'table', 'distance': 1.0, 'direction': 'center', 'private': False},
                            {'label': 'apple', 'distance': 1.1, 'direction': 'left',   'private': False},
                            {'label': 'photo', 'distance': 1.1, 'direction': 'center', 'private': True}],
               'hazard': None, 'navigable_floor': True, 'nearest_obstacle_dist': 1.0}},
    {'name': 'Staircase', 'query': None,
     'world': {'objects': [{'label': 'stairs', 'distance': 0.8, 'direction': 'center', 'private': False}],
               'hazard': 'stairs_down', 'navigable_floor': False, 'nearest_obstacle_dist': 0.8}},
    {'name': 'Find cup', 'query': 'find my cup',
     'world': {'objects': [{'label': 'cup',    'distance': 1.4, 'direction': 'right',  'private': False},
                            {'label': 'bottle', 'distance': 2.0, 'direction': 'left',   'private': False}],
               'hazard': None, 'navigable_floor': True, 'nearest_obstacle_dist': 1.4}},
    {'name': 'Person approaching', 'query': None,
     'world': {'objects': [{'label': 'person', 'distance': 1.5, 'direction': 'center', 'private': False},
                            {'label': 'wall',   'distance': 0.4, 'direction': 'left',   'private': False}],
               'hazard': None, 'navigable_floor': True, 'nearest_obstacle_dist': 1.5}},
]

print('=' * 60)
print('ARGUS LLM Scenario Tests')
print('=' * 60)
latencies = []
for i, s in enumerate(test_scenarios):
    print(f'\n[{i+1}] {s["name"]} | Query: {s["query"] or "(ambient navigation)"}')
    response, latency = world_model_to_speech(s['world'], s['query'])
    latencies.append(latency)
    print(f'     ARGUS says: "{response}"')
    print(f'     Latency: {latency:.0f}ms')

avg = sum(latencies) / len(latencies)
print(f'\nAverage latency: {avg:.0f}ms | Target: <400ms | {"PASSING" if avg < 400 else "TOO SLOW"}')

## Cell 06-06 | Optional QLoRA Fine-Tune on Custom Data

In [ ]:
# Optional: fine-tune Phi-3.5 with LoRA if base responses are not good enough.
# Skip this cell if Cell 06-05 results are acceptable.
import os, json, subprocess, sys, torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets'])

LORA_DIR  = f'{MODELS}/phi35_lora'
DONE_FLAG = f'{LORA_DIR}/DONE'
JSONL     = f'{BASE}/datasets/llm_finetune/train.jsonl'
os.makedirs(LORA_DIR, exist_ok=True)

if os.path.exists(DONE_FLAG):
    print(f'LoRA fine-tuning already complete — skipping.')
    print(f'Adapter at: {LORA_DIR}')
else:
    # ── Load dataset ──────────────────────────────────────────────────────────
    training_data = []
    if os.path.exists(JSONL):
        with open(JSONL, encoding='utf-8') as fh:
            training_data = [json.loads(l) for l in fh if l.strip()]
        print(f'Loaded {len(training_data)} examples from {JSONL}')
    else:
        # Built-in seed examples — add 500+ entries to JSONL for real fine-tuning
        training_data = [
            {'prompt': 'SCENE DATA: Nearest obstacle: 1.0m [Object: table, 1.0m, center]\nUser asks: what is ahead?',
             'completion': 'There is a table directly in front of you, one meter away.'},
            {'prompt': 'SCENE DATA: HAZARD: stairs_down Nearest obstacle: 0.8m\nAmbient navigation.',
             'completion': 'Caution! Stairs going down, about 80 centimeters ahead. Please stop.'},
            {'prompt': 'SCENE DATA: [Object: person, 2.0m, right] [Object: door_open, 1.5m, center]\nAmbient navigation.',
             'completion': 'Open door ahead, one and a half meters. Person to your right.'},
            {'prompt': 'SCENE DATA: No clear floor detected Nearest obstacle: 0.5m\nUser asks: can I walk forward?',
             'completion': 'I cannot see a clear path ahead. Something is about half a meter away — please stop.'},
        ]
        print(f'Using {len(training_data)} seed examples. Add more at: {JSONL}')

    if len(training_data) < 4:
        print('Not enough training data — skipping LoRA fine-tuning.')
    else:
        from transformers import (AutoModelForCausalLM, AutoTokenizer,
                                  BitsAndBytesConfig, TrainingArguments)
        from peft import LoraConfig, TaskType, get_peft_model
        from trl import SFTTrainer
        from datasets import Dataset

        MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'
        print(f'Loading {MODEL_ID} in 4-bit...')

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        model     = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=bnb_config,
            device_map='auto', trust_remote_code=True,
        )
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
            lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

        def fmt(ex):
            return (f"<|user|>\n{ex['prompt']}<|end|>\n"
                    f"<|assistant|>\n{ex['completion']}<|end|>")

        dataset = Dataset.from_dict({'text': [fmt(ex) for ex in training_data]})

        # ── Detect existing checkpoint for resume ─────────────────────────────
        ckpts = [d for d in os.listdir(LORA_DIR) if d.startswith('checkpoint-')]
        resume = True if ckpts else None
        if resume:
            print(f'Resuming LoRA training from checkpoint in {LORA_DIR}')

        training_args = TrainingArguments(
            output_dir=LORA_DIR,
            num_train_epochs=3,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            bf16=True,
            save_steps=25,
            save_total_limit=3,
            logging_steps=10,
            report_to='none',
            dataloader_num_workers=4,
        )
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
            dataset_text_field='text',
            max_seq_length=512,
        )
        trainer.train(resume_from_checkpoint=resume)
        trainer.save_model(LORA_DIR)
        tokenizer.save_pretrained(LORA_DIR)

        with open(DONE_FLAG, 'w') as fh:
            fh.write('done')
        print(f'LoRA adapter saved: {LORA_DIR}')
        print('To use: model.load_adapter(LORA_DIR) after loading base Phi-3.5')
